# MetFoundation: Embedding Extraction Demo

This notebook demonstrates how to extract embeddings from the pre-trained MetFoundation model using metabolomic data.

## Overview
- Load pre-trained model weights and tokenizer
- Process metabolomic data from h5ad format
- Extract sample embeddings for downstream analysis

## 1. Import Required Libraries

In [1]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import anndata as ad
from tqdm import tqdm
import pickle

# Add Src directory to path for importing custom modules
sys.path.append('./Src')

from metfoundation_torch.models import MetFoundation_Pretrained
from metfoundation_torch.tokenizer import MetFoundation_Tokenizer

from metfoundation_torch.dataset import load_dataset_from_adata_NMR
from utils import load_tokenizer,save_tokenizer, set_seeds
import json

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Libraries imported successfully!
PyTorch version: 2.3.0
CUDA available: False


## 2. Configuration

Set up paths and parameters for model loading and embedding extraction.

In [2]:
# Set random seed for reproducibility
set_seeds(3047)

# Configuration
model_dir = './Model_Weights/MetFoundation'  # Pre-trained model directory
data_path = './Data/UKB_NMR/val.h5ad'  # Data file path
batch_size = 128  # Batch size for processing
device = "cuda" if torch.cuda.is_available() else "cpu"  # Use GPU if available

print(f"Model directory: {model_dir}")
print(f"Data path: {data_path}")
print(f"Batch size: {batch_size}")
print(f"Device: {device}")

Model directory: ./Model_Weights/MetFoundation
Data path: /datahome/comp/ericteam/csyuxu/MetFoundation/Data/UKB_NMR/val.h5ad
Batch size: 128
Device: cpu


## 3. Load Data

Load the metabolomic data from the h5ad file and inspect its structure.

In [3]:
# Load the dataset from h5ad file
dataloader, adata = load_dataset_from_adata_NMR(
    data_path, 
    shuffle=False,
    batch_size=batch_size,
    device=device
)

print(f"Data loaded successfully!")
print(f"Number of samples: {adata.n_obs}")
print(f"Number of metabolites: {adata.n_vars}")
print(f"Number of batches: {len(dataloader)}")
print(f"\nData layers available: {list(adata.layers.keys())}")
print(f"Observation metadata columns: {list(adata.obs.columns)}")

Data loaded successfully!
Number of samples: 4320
Number of metabolites: 107
Number of batches: 34

Data layers available: ['Z-score normalized']
Observation metadata columns: ['Age at assessment (estimated)', 'Sex', 'Body mass index (BMI)', 'UK Biobank assessment centre', 'Date of attending assessment centre', 'Death follow-up time', 'Death event', 'Death event time', 'UK Biobank assessment area', 'instance']


## 4. Load Pre-trained Model

Load the tokenizer and pre-trained MetFoundation model.

In [5]:
# Load tokenizer
tokenizer_path = os.path.join(model_dir, 'tokenizer.pkl')
if not os.path.exists(tokenizer_path):
    raise FileNotFoundError(f"Tokenizer not found at: {tokenizer_path}")

# Tokenizer = load_tokenizer(tokenizer_path)

# tokenization setting
metabo_id = adata.var_names.values.tolist()

Tokenizer = MetFoundation_Tokenizer(VOCAB_Identifiers=metabo_id) 
# save tokenizer
save_tokenizer(Tokenizer,save_dir=model_dir)      

print(f"Tokenizer loaded successfully!")
print(f"Vocabulary size: {Tokenizer.vocab_size_identifiers}")

# Load model configuration
config_path = os.path.join(model_dir, 'config.json')
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Configuration file not found at: {config_path}")

with open(config_path, 'r') as f:
    model_config = json.load(f)

print(f"\nModel Configuration:")
print(f"- Model dimension: {model_config['d_model']}")
print(f"- Number of heads: {model_config['n_heads']}")
print(f"- Number of blocks: {model_config['n_blocks']}")
print(f"- Feed-forward dimension: {model_config['d_ff']}")

Tokenizer loaded successfully!
Vocabulary size: 107

Model Configuration:
- Model dimension: 512
- Number of heads: 8
- Number of blocks: 6
- Feed-forward dimension: 2048


In [7]:
device

'cpu'

In [8]:
# Prepare embedding module configuration
EmbeddingModule_conf = {
    "n_vocabs": {'identifier': Tokenizer.vocab_size_identifiers}
}

# Load model weights
model_weights_path = os.path.join(model_dir, 'model_weights.pth')
if not os.path.exists(model_weights_path):
    raise FileNotFoundError(f"Model weights not found at: {model_weights_path}")

# Initialize and load the pre-trained model
model = MetFoundation_Pretrained(EmbeddingModule_conf, model_config, model_weights_path)
model.to(device)
model.eval()  # Set to evaluation mode

print(f"Pre-trained model loaded successfully!")
print(f"Model moved to device: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Pre-trained model loaded successfully!
Model moved to device: cpu
Total parameters: 19,277,785


## 5. Extract Embeddings

Extract sample embeddings from the pre-trained model.

In [9]:
# Function to extract embeddings
def extract_embeddings(model, dataloader, tokenizer, data_layer='Z-score normalized', device='cpu'):
    """
    Extract sample embeddings from the pre-trained model.
    
    Args:
        model: Pre-trained MetFoundation model
        dataloader: DataLoader containing the metabolomic data
        tokenizer: Tokenizer for processing the data
        data_layer: Which layer of data to use for tokenization
        device: Computing device (cpu or cuda)
    
    Returns:
        embeddings: numpy array of sample embeddings
    """
    model.eval()
    embs_collect = []
    
    with torch.no_grad():
        for data in tqdm(dataloader, desc='Extracting embeddings'):
            # Tokenize the metabolomic data
            inputs, _ = tokenizer.tokenize_from_anndata(
                data, 
                padding='longest', 
                masking='missing',  # Mask missing values
                data_layer=data_layer, 
                mode='inference',
                return_tensor=True, 
                device=device
            )
            
            # Forward pass through the model
            outputs = model(inputs)
            
            # Extract sample embeddings (CLS token embeddings)
            embs = outputs['embs']
            embs_collect.append(embs.cpu().detach().numpy())
    
    # Concatenate all embeddings
    embeddings = np.concatenate(embs_collect, axis=0)
    return embeddings

# Extract embeddings
print("Starting embedding extraction...")
embeddings = extract_embeddings(model, dataloader, Tokenizer, device=device)

print(f"\nEmbedding extraction completed!")
print(f"Embeddings shape: {embeddings.shape}")
print(f"- Number of samples: {embeddings.shape[0]}")
print(f"- Embedding dimension: {embeddings.shape[1]}")

Starting embedding extraction...


Extracting embeddings:   0%|          | 0/34 [00:00<?, ?it/s]/home/comp/csyuxu/anaconda3/envs/MetFoundation/lib/python3.12/site-packages/sympy/external/gmpy.py:139: UserWarning: gmpy2 version is too old to use (2.0.0 or newer required)
  gmpy = import_module('gmpy2', min_module_version=_GMPY2_MIN_VERSION,
/home/comp/csyuxu/anaconda3/envs/MetFoundation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Extracting embeddings: 100%|██████████| 34/34 [00:46<00:00,  1.37s/it]


Embedding extraction completed!
Embeddings shape: (4320, 512)
- Number of samples: 4320
- Embedding dimension: 512


## 6. Analyze Embeddings

Perform basic statistical analysis on the extracted embeddings.

In [10]:
# Basic statistics of the embeddings
print("Embedding Statistics:")
print(f"Mean: {np.mean(embeddings):.4f}")
print(f"Standard deviation: {np.std(embeddings):.4f}")
print(f"Min value: {np.min(embeddings):.4f}")
print(f"Max value: {np.max(embeddings):.4f}")
print(f"Median: {np.median(embeddings):.4f}")

# Check for any NaN or Inf values
print(f"\nData quality checks:")
print(f"Contains NaN: {np.isnan(embeddings).any()}")
print(f"Contains Inf: {np.isinf(embeddings).any()}")

# Display sample embedding vector
print(f"\nFirst sample embedding (first 10 dimensions):")
print(embeddings[0, :10])

Embedding Statistics:
Mean: -0.0004
Standard deviation: 0.6936
Min value: -7.7427
Max value: 4.7078
Median: 0.0021

Data quality checks:
Contains NaN: False
Contains Inf: False

First sample embedding (first 10 dimensions):
[-0.0057873   0.22858395  0.31893536 -0.0834619  -0.22473045 -0.6933939
 -0.21447133 -0.14781398 -0.80754954  1.2260332 ]


## 7. Save Embeddings

Save the extracted embeddings for downstream analysis.

In [11]:
# Create output directory
output_dir = './Demo_output'
os.makedirs(output_dir, exist_ok=True)

# Save embeddings as numpy array
embeddings_path = os.path.join(output_dir, 'val_embeddings.npy')
np.save(embeddings_path, embeddings)
print(f"Embeddings saved as numpy array: {embeddings_path}")

# Save embeddings as pickle file (includes metadata)
embedding_dict = {
    'embeddings': embeddings,
    'sample_ids': adata.obs_names.tolist(),
    'n_samples': embeddings.shape[0],
    'embedding_dim': embeddings.shape[1],
    'model_config': model_config,
    'data_path': data_path
}

pickle_path = os.path.join(output_dir, 'val_embeddings.pkl')
with open(pickle_path, 'wb') as f:
    pickle.dump(embedding_dict, f)
print(f"Embeddings with metadata saved as pickle: {pickle_path}")

# Save embeddings as CSV (for easy viewing)
csv_path = os.path.join(output_dir, 'val_embeddings.csv')
emb_df = pd.DataFrame(
    embeddings, 
    index=adata.obs_names,
    columns=[f'dim_{i}' for i in range(embeddings.shape[1])]
)
emb_df.to_csv(csv_path)
print(f"Embeddings saved as CSV: {csv_path}")

print(f"\nAll files saved successfully in: {output_dir}")

Embeddings saved as numpy array: ./Demo_output/val_embeddings.npy
Embeddings with metadata saved as pickle: ./Demo_output/val_embeddings.pkl
Embeddings saved as CSV: ./Demo_output/val_embeddings.csv

All files saved successfully in: ./Demo_output


## Summary and Next Steps

### Summary
This notebook demonstrated how to:
1. Load metabolomic data from h5ad format
2. Initialize the pre-trained MetFoundation model
3. Extract sample embeddings using the pre-trained model
4. Save embeddings in multiple formats for downstream analysis

### Output Files
The extracted embeddings are saved in `./Extracted_Embeddings/`:
- `val_embeddings.npy` - NumPy array format (for efficient loading)
- `val_embeddings.pkl` - Pickle format with metadata
- `val_embeddings.csv` - CSV format (for easy viewing and analysis)

### Next Steps
You can use these embeddings for various downstream tasks:
- **Predictive modeling**: Use embeddings as features for ML models
- **Clustering analysis**: Group samples based on embedding similarity
- **Dimensionality reduction**: Visualize with UMAP, t-SNE, or PCA
- **Transfer learning**: Fine-tune the model for specific tasks
- **Association studies**: Link embeddings to phenotypes or outcomes

### Example: Loading Saved Embeddings
```python
# Load numpy format
embeddings = np.load('./Extracted_Embeddings/val_embeddings.npy')

# Load pickle format with metadata
with open('./Extracted_Embeddings/val_embeddings.pkl', 'rb') as f:
    data = pickle.load(f)
    embeddings = data['embeddings']
    sample_ids = data['sample_ids']
```

## Additional Information

### Data Format
The input data is expected to be in AnnData h5ad format with:
- **X**: Raw metabolite concentration values
- **layers['Z-score normalized']**: Z-score normalized values (used for model input)
- **obs**: Sample metadata
- **var**: Metabolite metadata

### Model Architecture
MetFoundation is a transformer-based model that:
- Uses a custom embedding layer for metabolites and concentrations
- Applies MixDirect masking for handling missing values
- Extracts a [CLS] token embedding as the sample representation

### Citation
If you use MetFoundation in your research, please cite the original paper.

---
**End of Demo Notebook**